In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, TargetEncoder
from sklearn.compose import ColumnTransformer

In [2]:
from credit_risk.dataset import AFTER_EDA, load_splits
from credit_risk.features import CATEGORICAL_COLS, prep_one_split, DROP_COLS, NUMERICAL_COLS

2026-06-14 16:19:07.516 | INFO     | credit_risk.config:<module>:11 - PROJ_ROOT path is: /Users/ak007/SML/Credit-Risk-Default-Prediction-System


In [3]:
from credit_risk.evaluation import evaluate_model

In [4]:
train_df, val_df, test_df, _ = load_splits(path=AFTER_EDA)

2026-06-14 16:19:08.125 | INFO     | credit_risk.dataset:load_splits:263 - Checking if the files exists...
2026-06-14 16:19:08.133 | INFO     | credit_risk.dataset:load_splits:265 - Loading the Cached files...
2026-06-14 16:19:08.475 | INFO     | credit_risk.dataset:load_splits:273 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466042, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)


In [5]:
# regime 2 dropping int_rate, grade and sub_grade
CATEGORICAL_COLS.remove('grade')
CATEGORICAL_COLS.remove('sub_grade')
NUMERICAL_COLS.remove('int_rate')

In [6]:
filtered_train_df, y_train = prep_one_split(df=train_df, drop_cols=DROP_COLS)
filtered_val_df, y_val = prep_one_split(df=val_df, drop_cols=DROP_COLS)
filtered_test_df, y_test = prep_one_split(df=test_df, drop_cols=DROP_COLS)

2026-06-14 16:19:10.707 | INFO     | credit_risk.features:prep_one_split:207 - Inside Function: prep_one_split
2026-06-14 16:19:10.708 | INFO     | credit_risk.features:split_target_and_features:143 - Inside Function: split_target_and_features
2026-06-14 16:19:10.708 | INFO     | credit_risk.features:split_target_and_features:144 - Splitting the target and the features...
2026-06-14 16:19:10.711 | INFO     | credit_risk.features:split_target_and_features:147 - features and target are splitted successfully...
2026-06-14 16:19:10.711 | INFO     | credit_risk.features:add_credit_yrs:160 - Inside Function: add_credit_yrs
2026-06-14 16:19:10.711 | INFO     | credit_risk.features:add_credit_yrs:162 - Adding the feature 'credit_yrs'
2026-06-14 16:19:10.727 | INFO     | credit_risk.features:add_credit_yrs:164 - 'credit_age_yrs added successfully!'
2026-06-14 16:19:10.727 | INFO     | credit_risk.features:add_fico_mid:169 - Inside Function: add_fico_mid
2026-06-14 16:19:10.727 | INFO     | cred

In [7]:
missing_per = ((filtered_train_df.isna().sum()/len(filtered_train_df))*100).sort_values(ascending=False)

In [9]:
missing_per.head(45)

mths_since_last_record            86.566232
mths_since_last_major_derog       78.772729
mths_since_recent_bc_dlq          78.765648
mths_since_recent_revol_delinq    70.149686
mths_since_last_delinq            53.690225
mths_since_recent_inq             19.750151
mo_sin_old_il_acct                17.926710
num_tl_120dpd_2m                  16.813935
pct_tl_nvr_dlq                    15.105935
avg_cur_bal                       15.075680
mo_sin_rcnt_rev_tl_op             15.073320
mo_sin_old_rev_tl_op              15.073320
num_il_tl                         15.073105
total_rev_hi_lim                  15.073105
tot_coll_amt                      15.073105
tot_cur_bal                       15.073105
num_bc_tl                         15.073105
mo_sin_rcnt_tl                    15.073105
num_op_rev_tl                     15.073105
num_actv_rev_tl                   15.073105
num_rev_accts                     15.073105
tot_hi_cred_lim                   15.073105
num_tl_30dpd                    

In [10]:
((filtered_train_df['tot_cur_bal'].isna()).astype(int) == (filtered_train_df['num_bc_tl'].isna()).astype(int)).all()

np.True_

In [11]:
MISSING_INFO = ['mths_since_last_record', 'mths_since_last_major_derog', 'mths_since_last_major_derog', 'tot_cur_bal']

In [12]:
num_pipeline = Pipeline([
    ('inpute', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())
])

cat_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

indicator_pipeline = Pipeline([
    ('ind', MissingIndicator(features='all'))
])

In [13]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, NUMERICAL_COLS),
    ('cat', cat_pipeline, CATEGORICAL_COLS),
    ('ind', indicator_pipeline, MISSING_INFO)
])

In [14]:
preprocessor.fit(X=filtered_train_df, y=y_train.to_numpy())

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e

In [15]:
X_train_feat = preprocessor.transform(filtered_train_df)
X_val_feat = preprocessor.transform(filtered_val_df)
X_test_feat = preprocessor.transform(filtered_test_df)

In [20]:
filtered_train_df.shape, filtered_val_df.shape, filtered_test_df.shape

((466042, 68), (420204, 68), (431712, 68))

In [21]:
X_train_feat.shape, X_val_feat.shape, X_test_feat.shape

((466042, 151), (420204, 151), (431712, 151))

In [16]:
from credit_risk.utils import to_jsonable, train

In [22]:
train_proba, val_proba, test_proba = train(
    X_train=X_train_feat,
    y_train=y_train.to_numpy(),
    X_val=X_val_feat,
    X_test=X_test_feat,
    model='LR'
)

In [23]:
lr_eval_dict = evaluate_model(
    y_train=y_train.to_numpy(),
    train_proba=train_proba,
    y_val=y_val.to_numpy(),
    val_proba=val_proba,
    y_test=y_test.to_numpy(),
    test_proba=test_proba,
    fn_cost=10000,
    fp_cost=2000
)

In [24]:
lr_eval_dict

{'threshold': np.float64(0.15000000000000002),
 'train': {'ROC-AUC': 0.6955223704141006,
  'PR-AUC': 0.3076489009640454,
  'brier_score': 0.12912362606826258,
  'precision': 0.24925175274357328,
  'recall': 0.7051227527546878,
  'confusion_matrix': array([[223648, 164799],
         [ 22881,  54714]])},
 'val': {'ROC-AUC': 0.7001120073577245,
  'PR-AUC': 0.3389629159300263,
  'brier_score': 0.13824964668272835,
  'precision': 0.27016138713224525,
  'recall': 0.7291447973769812,
  'confusion_matrix': array([[191047, 151994],
         [ 20900,  56263]])},
 'test': {'ROC-AUC': 0.687158114453226,
  'PR-AUC': 0.3015870640447048,
  'brier_score': 0.13145489180652853,
  'precision': 0.25082587663232825,
  'recall': 0.687326574905904,
  'confusion_matrix': array([[209465, 149449],
         [ 22762,  50036]])}}

In [25]:
train_proba, val_proba, test_proba = train(
    X_train=X_train_feat,
    y_train=y_train.to_numpy(),
    X_val=X_val_feat,
    X_test=X_test_feat,
    model='RF'
)

In [26]:
rf_eval_dict = evaluate_model(
    y_train=y_train.to_numpy(),
    train_proba=train_proba,
    y_val=y_val.to_numpy(),
    val_proba=val_proba,
    y_test=y_test.to_numpy(),
    test_proba=test_proba,
    fn_cost=10000,
    fp_cost=2000
)

In [27]:
rf_eval_dict

{'threshold': np.float64(0.18000000000000002),
 'train': {'ROC-AUC': 1.0,
  'PR-AUC': 1.0,
  'brier_score': 0.018317230635865437,
  'precision': 0.9664462130553376,
  'recall': 1.0,
  'confusion_matrix': array([[385753,   2694],
         [     0,  77595]])},
 'val': {'ROC-AUC': 0.681655539619585,
  'PR-AUC': 0.31437025198378393,
  'brier_score': 0.14048197613540092,
  'precision': 0.2631352850148158,
  'recall': 0.698560190765004,
  'confusion_matrix': array([[192095, 150946],
         [ 23260,  53903]])},
 'test': {'ROC-AUC': 0.6613063836222921,
  'PR-AUC': 0.27449436593651944,
  'brier_score': 0.13417194634385884,
  'precision': 0.23891704009821207,
  'recall': 0.6496194950410725,
  'confusion_matrix': array([[208266, 150648],
         [ 25507,  47291]])}}

In [28]:
train_proba, val_proba, test_proba = train(
    X_train=X_train_feat,
    y_train=y_train.to_numpy(),
    X_val=X_val_feat,
    X_test=X_test_feat,
    model='XGB'
)

In [29]:
xgb_eval_dict = evaluate_model(
    y_train=y_train.to_numpy(),
    train_proba=train_proba,
    y_val=y_val.to_numpy(),
    val_proba=val_proba,
    y_test=y_test.to_numpy(),
    test_proba=test_proba,
    fn_cost=10000,
    fp_cost=2000
)

In [30]:
xgb_eval_dict

{'threshold': np.float64(0.16),
 'train': {'ROC-AUC': 0.7605249823497228,
  'PR-AUC': 0.41415619212998167,
  'brier_score': 0.12007377296686172,
  'precision': 0.29070197387415225,
  'recall': 0.7396481732070366,
  'confusion_matrix': array([[248411, 140036],
         [ 20202,  57393]])},
 'val': {'ROC-AUC': 0.7019786680047024,
  'PR-AUC': 0.33464316680045,
  'brier_score': 0.13862502574920654,
  'precision': 0.280553396222451,
  'recall': 0.6922229565983696,
  'confusion_matrix': array([[206067, 136974],
         [ 23749,  53414]])},
 'test': {'ROC-AUC': 0.6863045415412978,
  'PR-AUC': 0.297590819446165,
  'brier_score': 0.1322914958000183,
  'precision': 0.25838678737948584,
  'recall': 0.6402098958762603,
  'confusion_matrix': array([[225147, 133767],
         [ 26192,  46606]])}}